In [29]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
import warnings

warnings.filterwarnings("ignore")

In [174]:
# leer csv's
caracteristicas_sitios = pd.read_csv("../../data/caracteristicas_sitios.csv")
consumo_trafico = pd.read_csv("../../data/consumos_trafico.csv")
# cambiar nombre de columns id_pop -> codigo 
consumo_trafico = consumo_trafico.rename(columns={'id_pop':'codigo'})
consumo_energia = pd.read_csv("../../data/consumos_energia.csv")

In [175]:
# hacer merge de caracteristicas sitios 
# generar columnas fecha id , month id en energia
consumo_energia['fecha_norm'] = pd.to_datetime(consumo_energia['date'])
consumo_energia['year_id'] = consumo_energia.fecha_norm.dt.year 
consumo_energia['month_id'] = consumo_energia.fecha_norm.dt.month

# revisar fechas minimas y maximas entre energia y trafico 
print(f"fecha minima trafico (anio/mes): {consumo_trafico.year_id.min()}-{consumo_trafico.month_id.min()}")
print(f"fecha minima energia (anio/mes): {consumo_energia.year_id.min()}-{consumo_energia.month_id.min()}")

# filtrar por el minimo de trafico 
consumo_energia = consumo_energia[consumo_energia.year_id >= consumo_trafico.year_id.min()]

# eliminar date y  fecha_norm
consumo_energia = consumo_energia.drop(['date','fecha_norm','tercera_categoria','consumo_normalizado'],axis=1)

# # generar cruce
df_merged = consumo_energia.merge(consumo_trafico, on=['codigo','year_id','month_id'],how='left')

fecha minima trafico (anio/mes): 2023-1
fecha minima energia (anio/mes): 2020-1


In [176]:
# eliminar valores nulos con subset todas las columnas de trafico 
df_merged = df_merged.dropna(subset=df_merged.columns[8:12])

# eliminar columna vol_total_gb 
df_merged = df_merged.drop(['vol_total_gb','vol_4gtdd_kmin','vol_5g_kmin'],axis=1)

# limpiar valores de caracteristicas
caracteristicas_sitios = caracteristicas_sitios.drop(['latitud','longitud','sistema_energia','id_pop','panel_solar'],axis=1)

# realizar merge con las caracteristicas
df_merged = df_merged.merge(caracteristicas_sitios, on='codigo',how='left')
df_merged = df_merged.dropna(subset=['categoria_planificacion','vol_2g_ul_gb','vol_2g_ul_gb'])
# rellenar valores de paneles como NO

# rellenar valores numericos con 0
df_cleaned = df_merged.fillna(0)

In [181]:
# guardar dataset final 
df_cleaned.to_csv('../../data/data_procesada_test.csv')